
# 📘 WikiLens 服务模块说明（文档版）

本说明汇总了 WikiLens 系统的全部服务模块，涵盖 Wikipedia 多语言内容的抓取、结构化分块、跨语言语义对齐与差异分析。每一部分的设计思路和功能在下方详细展开，适合项目文档归档与新成员阅读。

---

## 🧩 模块概览

| 模块名 | 功能简述 |
|--------|----------|
| `fetcher.py` | 抓取多语言 Wikipedia 页面，输出结构化文本（按章节、段落、句子） |
| `chunker.py` | 对抓取的句子进行语义相似性分块 |
| `embedder.py` | 对 chunk 和 section 生成语义嵌入向量 |
| `aligner_source_to_en.py` | 将各语言的 chunk 与英文 chunk 进行对齐 |
| `aligner_en_centered_structured.py` | 构建英文为中心的对齐结构 |
| `analyze_en_centered.py` | 输出每个英文 chunk 被对齐的详情（1:1, N:1, 0:1） |
| `analyze_source_to_en.py` | 输出每个源语言 chunk 的对齐详情（1:1, 1:N, 1:0） |


## 1. fetcher.py — 多语言抓取与断句

该模块负责从指定 Wikipedia URL 中抓取文章，并抽取其多语言版本。抓取内容保留原始结构（如小节层级），并对正文进行句子级别断句，输出结构化 JSON。

- 会跳过诸如“参考资料”、“外部链接”等非正文部分。
- 每条记录包含 `language`, `section`, `paragraph_index`, `sentence_index`, `sentence_text` 等字段。
- Section 会保留完整路径结构：如 “疫情发展 > 全球传播 > 欧洲”。

输出文件如：`fetched_zh.json`, `fetched_de.json` 等。


## 2. chunker.py — 相邻句语义分块

该模块对断句后的文本按段落进行句子语义分块（chunking）。

核心思想：
- 使用 SentenceTransformer（LaBSE）生成每句向量；
- 比较相邻句向量的相似度（余弦）；
- 若低于设定阈值（如 0.5），则认为出现话题切换，进行分块。

分块结果每句附有 `chunk_id`, `chunk_text`, `section`, `sentence_index` 等信息，便于后续嵌入与对齐。


## 3. embedder.py — 嵌入生成器

该模块将 `chunker` 的输出进行语义向量嵌入。

- 为每条 chunk 的正文（chunk_text）生成向量 `chunk_embedding`；
- 同时也为其路径（section）文本生成 `section_embedding`；
- 嵌入由 LaBSE 模型完成，输出 `.parquet` 格式，适配大规模处理。

该嵌入是后续对齐过程的基础特征。


## 4. aligner_source_to_en.py — 源语言 → 英语对齐

该模块对每种源语言与英文进行语义对齐。

工作流程：
- 使用 `chunk_embedding` 计算语义相似度；
- 使用 `section_embedding` 计算结构路径的语义相似度；
- 若路径相似度高，则优先在该路径内搜索候选对齐；否则 fallback 到全局；
- 结合 `文本相似度 + λ × 路径相似度` 得到 final score；
- 输出 match_type（1:1, 1:N, 1:0）及对应英文 chunks。

输出为：`structured_zh_to_en.json`, `structured_de_to_en.json` 等。


## 5. aligner_en_centered_structured.py — 英文为中心结构

该模块反向聚合各源语言对齐结果，以英文 chunk 为中心。

目的：
- 分析每个英文 chunk 是否被对齐；
- 若被多个 chunk 对齐 → `N:1`
- 若仅被一个对齐 → `1:1`
- 若未被任何 chunk 对齐 → `0:1`

输出：每个英文 chunk 的 `match_type` 和对应源语言 chunk 列表。

适用于 omission / amplification 分析。


## 6. analyze_en_centered.py — 英文视角分析

基于 `en_centered_from_{lang}.json`，该模块输出每个英文 chunk 在各语言下的详细对齐信息：

- `en_chunk_id`, `en_chunk_text`
- 对应源语言 chunk id / text
- match_type: `1:1`, `N:1`, `0:1`

每个语言生成一个 CSV 文件，可用于人工分析或偏见检测评估。


## 7. analyze_source_to_en.py — 源语言视角分析

该模块对源语言 chunk 进行反向分析，依据 `structured_{lang}_to_en.json` 生成：

- 每个源 chunk 的英文对齐对象
- match_type: `1:1`, `1:N`, `1:0`

用途：
- 检测 amplification（1:N）
- 检测过度压缩、缺失（1:0）

输出为 CSV，便于与英文视角数据交叉分析。


## 🧩 chunk_id 命名规则说明

每个 chunk 的唯一标识符采用以下格式进行命名：




### 字段说明：

| 字段名            | 说明                                                    |
|-------------------|---------------------------------------------------------|
| `language`        | 使用的语言代码（如 `en`, `zh`, `de`, `sw`, `eu`）        |
| `section_path`    | 所在小节的路径名（可能包含多层级，例如 `背景 > 起源`）    |
| `paragraph_index` | 该 chunk 所在的段落编号（从 0 开始）                    |
| `chunk_index`     | 段落内该 chunk 的编号（从 0 开始，3 位数字，如 `002`）  |

### 示例：

- `en_Origins > Lab Hypothesis_1_002`  
  表示英文中“Origins > Lab Hypothesis”这个小节下的第 2 段落（编号 1）中的第 3 个 chunk（编号 002）。

- `zh_发展历程 > 全球扩散_0_000`  
  表示中文中该路径下第 1 段第 1 个 chunk。


